# Personal Knowledge Decay Predictor

# Notebook 02

## Advanced Data Cleaning

---

### Objective

The purpose of this notebook is to prepare a clean and reliable interaction dataset for temporal analysis and feature engineering.

Tasks include:

- Missing value handling
- Removing irrelevant columns
- Optimizing data types
- Memory reduction
- Producing a cleaned dataset for the next stage

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_NAME = "Personal Knowledge Decay Predictor"

matches = [
    p for p in Path("/content/drive/MyDrive").rglob(PROJECT_NAME)
    if p.is_dir()
]

PROJECT_ROOT = matches[0]

sys.path.append(str(PROJECT_ROOT))

from config import *

✅ Project Root : /content/drive/MyDrive/Personal Knowledge Decay Predictor


🟦 Step 2 — Load Dataset

In [3]:
df = pd.read_csv(
    f"{DATA_DIR}/Interim/interactions_clean.csv",
    low_memory=False
)

print(df.shape)

(942816, 82)


🟦 Step 3 — Dataset Summary

In [4]:
summary = pd.DataFrame({
    "Data Type": df.dtypes,
    "Missing": df.isnull().sum(),
    "Missing %": (df.isnull().mean()*100).round(2),
    "Unique": df.nunique()
})

summary

,Data Type,Missing,Missing %,Unique
studentId,int64,0,0.00,1709
MiddleSchoolId,int64,0,0.00,4
InferredGender,object,173656,18.42,2
SY ASSISTments Usage,object,0,0.00,2
AveKnow,float64,0,0.00,1709
...,...,...,...,...
Ln,object,0,0.00,120113
MCAS,int64,0,0.00,52
Enrolled,int64,0,0.00,2
Selective,int64,0,0.00,2


🟦 Step 4 — Visualize Missing Values

In [5]:
missing = (
    df.isnull()
      .mean()
      .sort_values(ascending=False)
      *100
)

missing = missing[missing > 0]

missing

,0
isSTEM,66.380078
InferredGender,18.418864
sumTime3SDWhen3RowRight,0.009016


Step 5

Create Missing Value Categories.

In [6]:
missing_table = pd.DataFrame({
    "Feature": missing.index,
    "Missing %": missing.values
})

def classify_missing(x):

    if x == 0:
        return "Complete"

    elif x < 5:
        return "Low"

    elif x < 30:
        return "Moderate"

    elif x < 60:
        return "High"

    else:
        return "Very High"

missing_table["Category"] = missing_table["Missing %"].apply(classify_missing)

missing_table

,Feature,Missing %,Category
0,isSTEM,66.380078,Very High
1,InferredGender,18.418864,Moderate
2,sumTime3SDWhen3RowRight,0.009016,Low


# Step 6

## Remove Irrelevant Features

Based on the Feature Audit Catalog created in Notebook 01.5, the following features are removed because:

- They are not useful for Knowledge Decay Prediction.
- They introduce demographic bias.
- They belong to the original competition objective.

In [7]:
drop_columns = [

    "isSTEM",

    "InferredGender",

    "MiddleSchoolId",

    "Enrolled",

    "Selective"

]

# Drop only if they exist
df.drop(columns=drop_columns,
        inplace=True,
        errors="ignore")

print(df.shape)

(942816, 77)


🟦 Step 7 — Handle Remaining Missing Values
Now we only have

sumTime3SDWhen3RowRight

with

0.009%

missing.

That is extremely small.

In [8]:
print(df["sumTime3SDWhen3RowRight"].describe())

count    942731.000000
mean          0.087042
std           1.619202
min         -11.332080
25%           0.000000
50%           0.000000
75%           0.000000
max          92.709045
Name: sumTime3SDWhen3RowRight, dtype: float64


In [9]:
# Keep the feature numeric
df["sumTime3SDWhen3RowRight"] = pd.to_numeric(
    df["sumTime3SDWhen3RowRight"],
    errors="coerce"
)

# Fill the very small number of missing values
df["sumTime3SDWhen3RowRight"] = (
    df["sumTime3SDWhen3RowRight"]
      .fillna(df["sumTime3SDWhen3RowRight"].median())
)

🟦 Step 8 — Verify Missing Values

In [10]:
missing_after = df.isnull().sum()

missing_after[missing_after > 0]

,0


🟦 Step 9 — Data Type Optimization


In [11]:
before = df.memory_usage(deep=True).sum() / 1024**2

print(f"Before Optimization : {before:.2f} MB")

Before Optimization : 790.16 MB


Optimize integers

In [12]:
for col in df.select_dtypes(include="int").columns:

    df[col] = pd.to_numeric(
        df[col],
        downcast="integer"
    )

Optimize floats

In [13]:
for col in df.select_dtypes(include="float").columns:

    df[col] = pd.to_numeric(
        df[col],
        downcast="float"
    )

Check

In [14]:
after = df.memory_usage(deep=True).sum()/1024**2

print(f"After Optimization : {after:.2f} MB")

After Optimization : 444.90 MB


🟦 Step 10 — Cleaning Summary

In [15]:
cleaning_report = pd.DataFrame({

    "Metric":[

        "Rows",

        "Columns",

        "Missing Values",

        "Duplicate Rows"

    ],

    "Value":[

        len(df),

        len(df.columns),

        df.isnull().sum().sum(),

        df.duplicated().sum()

    ]

})

cleaning_report

,Metric,Value
0,Rows,942816
1,Columns,77
2,Missing Values,0
3,Duplicate Rows,0


🟦 Step 11 — Save Dataset

In [16]:
import os

print(os.getcwd())

/content/drive/.shortcut-targets-by-id/1LWX90DlmGF9QpN_sG10QqlUaqgqBZdLn/Personal Knowledge Decay Predictor


In [17]:
os.makedirs(
    "Data/Processed",
    exist_ok=True
)

In [18]:
df.to_csv(

    "Data/Processed/cleaned_dataset.csv",

    index=False

)

print("Dataset Saved Successfully")

Dataset Saved Successfully


In [19]:
df["sumTime3SDWhen3RowRight"].value_counts().head(20)

,count
sumTime3SDWhen3RowRight,
0.000000,856340
-2.644413,2
4.657585,2
-2.980790,2
5.541756,2
-1.210920,2
-0.705841,2
-4.834811,2
9.516756,2


In [20]:

df[df["sumTime3SDWhen3RowRight"] < 0].head()



,studentId,SY ASSISTments Usage,AveKnow,AveCarelessness,AveCorrect,NumActions,AveResBored,AveResEngcon,AveResConf,AveResFrust,...,confidence(GAMING),RES_BORED,RES_CONCENTRATING,RES_CONFUSED,RES_FRUSTRATED,RES_OFFTASK,RES_GAMING,Ln-1,Ln,MCAS
21,8,2004-2005,0.352416,0.183276,0.483902,1056,0.208389,0.679126,0.115905,0.112408,...,0.047821,0.156027,0.920388,0.0,0.009561,0.468252,0.001483,0.327087357,0.674752873,45
33,8,2004-2005,0.352416,0.183276,0.483902,1056,0.208389,0.679126,0.115905,0.112408,...,0.047821,0.156027,0.887818,0.0,0.009561,0.620180,0.001483,0.329185346,0.63061613,45
34,8,2004-2005,0.352416,0.183276,0.483902,1056,0.208389,0.679126,0.115905,0.112408,...,0.127084,0.156027,0.721888,0.0,0.009561,0.122595,0.003940,0.63061613,0.850947135,45
35,8,2004-2005,0.352416,0.183276,0.483902,1056,0.208389,0.679126,0.115905,0.112408,...,0.127084,0.156027,0.721888,0.0,0.009561,0.122595,0.003940,0.850947135,0.949682058,45
36,8,2004-2005,0.352416,0.183276,0.483902,1056,0.208389,0.679126,0.115905,0.112408,...,0.047821,0.156027,0.335618,0.0,0.009561,0.468252,0.001483,0.949682058,0.984172356,45


In [21]:
df[df["sumTime3SDWhen3RowRight"] < 0].describe()

,studentId,AveKnow,AveCarelessness,AveCorrect,NumActions,AveResBored,AveResEngcon,AveResConf,AveResFrust,AveResOfftask,...,confidence(FRUSTRATED),confidence(OFF TASK),confidence(GAMING),RES_BORED,RES_CONCENTRATING,RES_CONFUSED,RES_FRUSTRATED,RES_OFFTASK,RES_GAMING,MCAS
count,47695.00000,47695.000000,47695.000000,47695.000000,47695.000000,47695.000000,47695.000000,47695.000000,47695.000000,47695.000000,...,47695.000000,47695.000000,47695.000000,47695.000000,47695.000000,47695.000000,47695.000000,47695.000000,47695.000000,47695.000000
mean,3894.53037,0.299920,0.159912,0.469409,748.591131,0.235934,0.660104,0.098016,0.129262,0.181434,...,0.179606,0.279541,0.181458,0.181666,0.733062,0.004210,0.101951,0.177271,0.009876,-104.410148
std,2237.46515,0.156832,0.078972,0.138783,447.893140,0.025024,0.024798,0.031631,0.046244,0.048615,...,0.262537,0.173799,0.100247,0.091101,0.101282,0.059423,0.268418,0.183371,0.051831,353.284315
min,8.00000,0.029265,0.009858,0.184874,19.000000,0.170871,0.480271,0.005075,0.001510,0.083167,...,0.000000,0.000000,0.000039,0.156027,0.000738,0.000000,0.000000,0.000000,0.000001,-999.000000
25%,1977.00000,0.173345,0.098004,0.360087,426.000000,0.218780,0.645875,0.077379,0.101595,0.148691,...,0.091463,0.230769,0.127084,0.156027,0.719044,0.000000,0.009561,0.122595,0.003940,19.000000
50%,3835.00000,0.273180,0.146905,0.449871,648.000000,0.235211,0.661393,0.096790,0.126788,0.174748,...,0.091463,0.230769,0.186970,0.156027,0.728079,0.000000,0.009561,0.122595,0.005797,34.000000
75%,5859.00000,0.386930,0.201753,0.555024,971.000000,0.252201,0.677119,0.117727,0.158224,0.207775,...,0.091463,0.230769,0.186970,0.156027,0.772610,0.000000,0.009561,0.122595,0.005797,44.000000
max,7783.00000,0.752498,0.430576,0.932990,3057.000000,0.345638,0.723990,0.370515,0.543463,0.610001,...,1.000000,1.000000,0.999610,0.505313,0.997856,1.000000,1.000000,1.000000,0.999252,54.000000


In [22]:
df["sumTime3SDWhen3RowRight"].isna().sum()

np.int64(0)